# CcMart — Spark Structured Streaming Demo
**ITCS 6190/8190 Cloud Computing for Data Analysis**

Demonstrates real-time clickstream event processing using Spark Structured Streaming with windowed aggregations.

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, count, avg, from_json
from pyspark.sql.types import StructType, StringType, DoubleType, TimestampType

spark = SparkSession.builder \\
    .appName('CcMart-Streaming') \\
    .config('spark.sql.streaming.checkpointLocation', '../data/stream_checkpoint') \\
    .getOrCreate()
print('Streaming session ready. isStreaming preview below.')

## Define Clickstream Event Schema

In [ ]:
schema = StructType() \\
    .add('customer_id', StringType()) \\
    .add('session_id', StringType()) \\
    .add('event_type', StringType()) \\
    .add('product_id', StringType()) \\
    .add('event_time', TimestampType()) \\
    .add('session_duration', DoubleType()) \\
    .add('device', StringType()) \\
    .add('page_views', DoubleType())

print('Schema fields:', schema.fieldNames())

## Read Streaming JSON Files

In [ ]:
stream_df = spark.readStream \\
    .schema(schema) \\
    .json('../data/stream_input/')

print('isStreaming:', stream_df.isStreaming)
stream_df.printSchema()

## 5-Minute Windowed Aggregation

In [ ]:
# Count events per type in 5-minute windows
windowed_agg = stream_df \\
    .withWatermark('event_time', '10 minutes') \\
    .groupBy(
        window('event_time', '5 minutes'),
        'event_type'
    ) \\
    .agg(
        count('*').alias('event_count'),
        avg('session_duration').alias('avg_session_sec')
    )

print('Windowed aggregation query defined.')

## Write Output to Console (Demo)
> In production this writes to Parquet or Kafka
```python
# Run in production:
query = windowed_agg.writeStream \
    .outputMode('update') \
    .format('console') \
    .option('truncate', False) \
    .start()
query.awaitTermination(60)  # run for 60 seconds
```


## Concept: Why Structured Streaming?

| Batch (nightly) | Streaming (real-time) |
|---|---|
| React **next day** | React **while customer is on site** |
| Miss cart-abandon opportunity | Trigger 10% promo in same session |
| Report-driven | Action-driven |

Spark Structured Streaming treats data as an **unbounded table** — the same SQL you write for batch works on live data.

## Run the full streaming pipelines
```bash
python ../src/streaming.py
python ../src/streaming_predictions.py
```
Outputs saved to `outputs/streaming/`